In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import pyspark
import dxpy
import dxdata

# Read data

In [ ]:
# Clinical data
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

dispensed_database_name = dxpy.find_one_data_object(classname = 'database', name = 'app*', folder = '/', name_mode = 'glob', describe = True)['describe']['name']
dispensed_dataset_id = dxpy.find_one_data_object(typename = 'Dataset', name = 'app*.dataset', folder = '/', name_mode = 'glob')['id']

dataset = dxdata.load_dataset(id = dispensed_dataset_id)
participant = dataset['participant']

field_names = ['eid', 'p21022']

df = participant.retrieve_fields(names = field_names, engine = dxdata.connect())

inds_sql_df = df.toPandas()
inds_sql_df['eid'] = inds_sql_df['eid'].astype(int)
inds_sql_df = inds_sql_df.dropna(subset = ['p21022'])
inds_sql_df.head()

In [ ]:
# Array data
inds_gt_df = pd.read_csv('/mnt/project/DNM/snipar/QC/inds_post_QC.txt', sep = ' ', header = None)
inds_gt_df.columns = ['FID', 'IID']
inds_gt_df.head()

In [ ]:
# Individual-level WGS data
inds_gvcfs = {}
path = '/mnt/project/Bulk/DRAGEN WGS/Whole genome variant call files (GVCFs) (DRAGEN) [500k release]'

for d in tqdm(os.listdir(path)):
    for f in os.listdir(os.path.join(path, d)):
        inds_gvcfs[int(f.split('_')[0])] = True
        
print(list(inds_gvcfs.items())[:5])

In [ ]:
# Population-level WGS data
inds_plink_df = pd.read_csv('/mnt/project/Bulk/DRAGEN WGS/DRAGEN population level WGS variants, PLINK format [500k release]/ukb24308_c1_b0_v1.psam',
                            sep = ' ', header = None)
inds_plink_df.columns = ['FID', 'IID', 'PAT', 'MAT', 'SEX', 'REDACTED']

inds_plink_df = inds_plink_df[inds_plink_df['SEX'].isin([1, 2])]
inds_plink_df = inds_plink_df[inds_plink_df['REDACTED'] != 'redacted']
inds_plink_df = inds_plink_df[inds_plink_df['IID'].isin(inds_sql_df['eid'])]
inds_plink_df = inds_plink_df[inds_plink_df['IID'].isin(inds_gt_df['IID'])]
inds_plink_df = inds_plink_df[inds_plink_df['IID'].isin(inds_gvcfs)]

inds_plink_df.head()

In [ ]:
ind_age = inds_sql_df.set_index('eid')['p21022'].to_dict()
ind_sex = inds_plink_df.set_index('IID')['SEX'].to_dict()

In [ ]:
len(inds_plink_df)

In [ ]:
# Kinship data
inds_kin_df = pd.read_csv('/mnt/project/DNM/snipar/king.kin0', sep = '\t')
inds_kin_df.head()

In [ ]:
len(inds_kin_df)

# List related pairs

In [ ]:
twins_pairs = {}
for _, row in inds_kin_df.iterrows():
    
    ind1 = row['ID1']
    ind2 = row['ID2']
    rel = row['InfType']
    
    if (ind1 in inds_plink_df['IID'].values and 
        ind2 in inds_plink_df['IID'].values and 
        rel == 'Dup/MZ'):
        twins_pairs[(min(ind1, ind2), max(ind1, ind2))] = True
print('Pairs:', len(twins_pairs), list(twins_pairs.items())[:5])

twins_inds = {ind: True for pair in twins_pairs for ind in pair}
print('Individuals:', len(twins_inds))

In [ ]:
ops_pairs = {}
for _, row in inds_kin_df.iterrows():
    
    ind1 = row['ID1']
    ind2 = row['ID2']
    rel = row['InfType']
    
    if (ind1 in inds_plink_df['IID'].values and 
        ind2 in inds_plink_df['IID'].values and 
        rel == 'PO'):

        if ind_age[ind1] > ind_age[ind2]:
            if ind2 not in ops_pairs:
                ops_pairs[ind2] = {}
            ops_pairs[ind2][ind_sex[ind1]] = ind1
                
        elif ind_age[ind2] > ind_age[ind1]:
            if ind1 not in ops_pairs:
                ops_pairs[ind1] = {}
            ops_pairs[ind1][ind_sex[ind2]] = ind2
print('Pairs:', len(ops_pairs), list(ops_pairs.items())[:5])

In [ ]:
trios_pairs = {k: v for k, v in ops_pairs.items() if len(v) == 2}
print('Pairs:', len(trios_pairs), list(trios_pairs.items())[:5])

trios_inds = {ind: True for o, p in trios_pairs.items() for ind in [o, p[1], p[2]]}
print('Individuals:', len(trios_inds))

In [ ]:
sibs_pairs = {}
for _, row in inds_kin_df.iterrows():
    
    ind1 = row['ID1']
    ind2 = row['ID2']
    rel = row['InfType']
    
    if (ind1 in inds_plink_df['IID'].values and 
        ind2 in inds_plink_df['IID'].values and 
        rel == 'FS'):
        sibs_pairs[(min(ind1, ind2), max(ind1, ind2))] = True    
print('Pairs:', len(sibs_pairs), list(sibs_pairs.items())[:5])

sibs_inds = {ind: True for pair in sibs_pairs for ind in pair}
print('Individuals:', len(sibs_inds))

In [ ]:
quads_pairs = {k: trios_pairs[k[0]] for k, v in sibs_pairs.items() if (k[0] in trios_pairs and 
                                                                       k[1] in trios_pairs and 
                                                                       trios_pairs[k[0]] == trios_pairs[k[1]])}
print('Pairs:', len(quads_pairs), list(quads_pairs.items())[:5])

quads_inds = {ind: True for (o1, o2), p in quads_pairs.items() for ind in [o1, o2, p[1], p[2]]}
print('Individuals:', len(quads_inds))

In [ ]:
duo_pairs = {k: v for k, v in ops_pairs.items() if len(v) == 1 and k in sibs_inds}
print('Pairs:', len(duo_pairs), list(duo_pairs.items())[:5])

In [ ]:
o_pairs = {}

for o in ops_pairs:
    for i in ops_pairs[o]:
        if ops_pairs[o][i] in ops_pairs:
            if ops_pairs[o][i] not in o_pairs:
                o_pairs[ops_pairs[o][i]] = []
            o_pairs[ops_pairs[o][i]].append(o)
        elif ops_pairs[o][i] in sibs_inds:
            if ops_pairs[o][i] not in o_pairs:
                o_pairs[ops_pairs[o][i]] = []
            o_pairs[ops_pairs[o][i]].append(o)
            
len(o_pairs)

In [ ]:
grand_pairs = {}

for o1 in ops_pairs:
    for i in ops_pairs[o1]:
        for o2 in ops_pairs:
            if o2 == ops_pairs[o1][i]:
                grand_pairs[o1][i] = ops_pairs[o2].copy()
                
grand_pairs

# Save lists

In [ ]:
with open('inds.txt', 'w') as f:
    for ind in inds_plink_df['IID']:
        f.write(f'{ind} {ind}\n')

In [ ]:
with open('twins_pairs.txt', 'w') as f:
    for ind1, ind2 in twins_pairs:
        f.write(f'{ind1} {ind2}\n')
        
with open('twins_inds.txt', 'w') as f:
    for ind in twins_inds:
        f.write(f'{ind} {ind}\n')

In [ ]:
with open('trios_pairs.txt', 'w') as f:
    for ind in trios_pairs:
        f.write(f'{ind} {trios_pairs[ind][1]} {trios_pairs[ind][2]}\n')
        
with open('trios_inds.txt', 'w') as f:
    for ind in trios_inds:
        f.write(f'{ind} {ind}\n')

In [ ]:
with open('sibs_pairs.txt', 'w') as f:
    for ind1, ind2 in sibs_pairs:
        f.write(f'{ind1} {ind2}\n')
        
with open('sibs_inds.txt', 'w') as f:
    for ind in sibs_inds:
        f.write(f'{ind} {ind}\n')
        
with open('sibs_agesex.txt', 'w') as f:
    f.write('FID IID sex age\n')
    for ind in sibs_inds:
        sex = 'M' if ind_sex[ind] == 1 else 'F'
        f.write(f'{ind} {ind} {sex} {ind_age[ind]}\n')

In [ ]:
with open('quads_pairs.txt', 'w') as f:
    for ind1, ind2 in quads_pairs:
        f.write(f'{ind1} {ind2} {quads_pairs[(ind1, ind2)][1]} {quads_pairs[(ind1, ind2)][2]}\n')
        
with open('quads_inds.txt', 'w') as f:
    for ind in quads_inds:
        f.write(f'{ind} {ind}\n')

In [ ]:
with open('duo_pairs.txt', 'w') as f:
    for o in duo_pairs:
        for i in duo_pairs[o]:
            f.write(f'{o} {duo_pairs[o][i]}\n')

In [ ]:
with open('o_pairs.txt', 'w') as f:
    for o1 in o_pairs:
        f.write(f'{o1} ')
        for o2 in o_pairs[o1]:
            f.write(f'|{o2}')
        f.write('\n')

In [ ]:
!dx upload *.txt --path='DNM/'